In [ ]:
import numpy as np
import pandas as pd
import os
import glob

In [ ]:
os.getcwd()

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv('C:\\Users\\...\\merged_ready_to_delete_dup_geno_calls.csv')
df.head()

In [ ]:
print(df.shape)

In [ ]:
df.dtypes
# dtype 'object' is used for strings
# int64 for 64bit integers

In [ ]:
# change dtype of Sample Number to object
df['Sample Number'] = df['Sample Number'].astype(str)
df['NHS Number'] = df['NHS Number'].astype(str)

In [ ]:
df.dtypes

In [ ]:
# NOTE that the sample numbers have had the preceding 0 stripped at read_csv
# So adding it back in here
df['Sample Number'] = '0' + df['Sample Number'].astype(str)

In [ ]:
df.head()

In [ ]:
# get column header names 
column_names = list(df.columns.values)
print(column_names)

In [ ]:
# count the number of samples with the same genotype combinations in RHD
counts_incRH01G = df[['RHD_INT4', 'EX5', 'RHD455', 'RHD667', 'RH_EX7', 'RH01G']].value_counts().reset_index(name='count')
counts_incRH01G

In [ ]:
# counts_incRH01G.to_csv('counts_incRH01G', index=True)

In [ ]:
# in the above row 11 (count 17) and row 14 (count 13) are the same genotype with different conclusions of RH01G
# need to overcome this...
# for now, run the numbers in the absence of RH01G
counts_notRH01G = df[['RHD_INT4', 'EX5', 'RHD455', 'RHD667', 'RH_EX7']].value_counts().reset_index(name='count')
counts_notRH01G

In [ ]:
# counts_notRH01G.to_csv('counts_notRH01G', index=True)

In [ ]:
# counts_incRH01G = 51 unique combinations, 
# counts_notRH01G = 40 unique combinations
# suggests 11 pairs that have same geno but different predicted pheno
# see that row 11 has count 30.  This is 17+13.
# need to overcome this...

In [ ]:
# indexes below relate tot he index in counts_incRH01G
# this is important as it is these rows that have the same genotype but different 
# predicted phenotype
# VERY IMPORTANT to make sure the predicted phenotype is correct!!

In [ ]:
# one method is to re-translate all the genotypes into phenotypes
# row 0 of counts_incRH01G
# 0	AMP	0/0	0/0	0/0	two	P	2894
df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '0/0') &
                   (df['RHD455'] == '0/0') & 
                   (df['RHD667'] == '0/0') &
                   (df['RH_EX7'] == 'two'), 'P', np.nan)
# this creates a new column ['D'] and adds 'P' when the condition met and NaN when it is not

In [ ]:
Dcounts = df['D'].value_counts()
Dcounts
# row 0 = 2894 P

In [ ]:
# 1	AMP	0	0	0	one	P	1611
# 37	AMP	0	0	0	one	UND	1
df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '0') &
                   (df['RHD455'] == '0') & 
                   (df['RHD667'] == '0') &
                   (df['RH_EX7'] == 'one'), 'P', df['D'])
# df['D'] tells pandas to pass over the column if the condition is not met - clever!

In [ ]:
Dcounts = df['D'].value_counts()
Dcounts

In [ ]:
2960-1348
# the correct fix has decreased the number of nan calls by 1612

In [ ]:
# whilst this is working, setting up 44 conditional formats correctly seems time consuming
# and error prone.  It's also not future-proof as a 45th genuine combination may
# exist in future datasets

# I need a method that compares the 'counts' column in counts_incRH01G and counts_notRH01G
# if they are the same = great, if they are different go away and find the rows with the 
# same combinations of results in columns RHD_INT4, EX5, RHD455, RHD667 and RH_EX7
# and then what?!?!?!
# adding one UND to 1611 P results is fairly obvious (to a human) but if this were programatically
# possible how would I be reassured that the correct predicted phenotyep had been arrived at?
# manual check.
# so... maybe just write them all out

# maybe not as a correction - which is what has been done above
# maybe as a logic...

In [ ]:
# 2	AMP	0/1	0/0	0/0	two	P	249
df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '0/1') &
                   (df['RHD455'] == '0/0') & 
                   (df['RHD667'] == '0/0') &
                   (df['RH_EX7'] == 'two'), 'P', df['D'])
df['D'].value_counts()

In [ ]:
1348-1099

In [ ]:
# row 3 AMP 	0 	0/1 	0 	one 	230
# This combination of D results is only called by also considernig RHCE1006
# So it must in fact be split into three
# retaining original cell here for safe measure

#df['D'] = np.where((df['RHD_INT4'] =='AMP') &
#                   (df['EX5'] == '0') &
#                   (df['RHD455'] == '0/1') & 
#                   (df['RHD667'] == '0') &
#                   (df['RH_EX7'] == 'one'), 'P', df['D'])
# df['D'].value_counts()

In [ ]:
# 3	AMP	0	0/1	0	one	P	229
# 49	AMP	0	0/1	0	one	UND	1
# inc RHCE1006
df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '0') &
                   (df['RHD455'] == '0/1') & 
                   (df['RHD667'] == '0') &
                   (df['RH_EX7'] == 'one') &
                   (df['RHCE1006'] == '0/0'), 'UND', df['D'])
df['D'].value_counts()

In [ ]:
# 1 UND

In [ ]:
# row 3 AMP 	0 	0/1 	0 	one 	230
# inc RHCE1006
df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '0') &
                   (df['RHD455'] == '0/1') & 
                   (df['RHD667'] == '0') &
                   (df['RH_EX7'] == 'one') &
                   (df['RHCE1006'] == '0/1'), 'P', df['D'])
df['D'].value_counts()

In [ ]:
1098-869
# 229 P

In [ ]:
# row 3 AMP 	0 	0/1 	0 	one 	230
# inc RHCE1006
df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '0') &
                   (df['RHD455'] == '0/1') & 
                   (df['RHD667'] == '0') &
                   (df['RH_EX7'] == 'one') &
                   (df['RHCE1006'] == '1/1'), 'P', df['D'])
df['D'].value_counts()

In [ ]:
# none changed when RHCE1006 1/1
# total changed for index 3 = 229, index 49 = 1

In [ ]:
# 4 	NOAMP 	NOAMP 	NOAMP 	NOAMP 	zero 	N 	203
df['D'] = np.where((df['RHD_INT4'] =='NOAMP') &
                   (df['EX5'] == 'NOAMP') &
                   (df['RHD455'] == 'NOAMP') & 
                   (df['RHD667'] == 'NOAMP') &
                   (df['RH_EX7'] == 'zero'), 'N', df['D'])
df['D'].value_counts()

In [ ]:
869-666

In [ ]:
# 5 	AMP 	0/0 	0/0 	0/1 	two 	P 	162
df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '0/0') &
                   (df['RHD455'] == '0/0') & 
                   (df['RHD667'] == '0/1') &
                   (df['RH_EX7'] == 'two'), 'P', df['D'])
df['D'].value_counts()

In [ ]:
666-504

In [ ]:
# 6 	AMP 	0/0 	0/1 	0/0 	two 	P 	93
df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '0/0') &
                   (df['RHD455'] == '0/1') & 
                   (df['RHD667'] == '0/0') &
                   (df['RH_EX7'] == 'two'), 'P', df['D'])
df['D'].value_counts()

In [ ]:
504-411

In [ ]:
# 7 AMP 	0/0 	0/1 	0/1 	two 	VAR 	93
df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '0/0') &
                   (df['RHD455'] == '0/1') & 
                   (df['RHD667'] == '0/1') &
                   (df['RH_EX7'] == 'two'), 'VAR', df['D'])
df['D'].value_counts()

In [ ]:
411-318

In [ ]:
# 8 	NOAMP 	NOAMP 	1/1 	NOAMP 	zero 	N 	59
df['D'] = np.where((df['RHD_INT4'] =='NOAMP') &
                   (df['EX5'] == 'NOAMP') &
                   (df['RHD455'] == '1/1') & 
                   (df['RHD667'] == 'NOAMP') &
                   (df['RH_EX7'] == 'zero'), 'N', df['D'])
df['D'].value_counts()

In [ ]:
318-259

In [ ]:
# 9 	AMP 	1 	0 	NOAMP 	one 	N 	51
# 29	AMP	1	0	NOAMP	one	UND	2
df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '1') &
                   (df['RHD455'] == '0') & 
                   (df['RHD667'] == 'NOAMP') &
                   (df['RH_EX7'] == 'one'), 'N', df['D'])
df['D'].value_counts()

In [ ]:
259-206

In [ ]:
# 10 	AMP 	0 	0 	1 	one 	VAR 	36
# 47	AMP	0	0	1	one	UND	1
df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '0') &
                   (df['RHD455'] == '0') & 
                   (df['RHD667'] == '1') &
                   (df['RH_EX7'] == 'one'), 'VAR', df['D'])
df['D'].value_counts()

In [ ]:
206-169

In [ ]:
# 11 	AMP 	0 	1 	1 	one 	VAR 	17
# 14	AMP	0	1	1	one	UND	13
df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '0') &
                   (df['RHD455'] == '1') & 
                   (df['RHD667'] == '1') &
                   (df['RH_EX7'] == 'one'), 'VAR', df['D'])
df['D'].value_counts()

In [ ]:
169-139

In [ ]:
# 12 	AMP 	0/0 	1/1 	0/0 	two 	VAR 	16
# 41 	AMP 	0/0 	1/1 	0/0 	two 	P 	1
df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '0/0') &
                   (df['RHD455'] == '1/1') & 
                   (df['RHD667'] == '0/0') &
                   (df['RH_EX7'] == 'two'), 'VAR', df['D'])
df['D'].value_counts()

In [ ]:
139-122

In [ ]:
# 13 	AMP 	0/0 	0/0 	1/1 	two 	VAR 	14
df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '0/0') &
                   (df['RHD455'] == '0/0') & 
                   (df['RHD667'] == '1/1') &
                   (df['RH_EX7'] == 'two'), 'VAR', df['D'])
df['D'].value_counts()

In [ ]:
122-108

In [ ]:
# 14 of counts_incRH01G was mothballed into 11 - correctly

# 15 	AMP 	0 	1 	0 	one 	VAR 	11
df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '0') &
                   (df['RHD455'] == '1') & 
                   (df['RHD667'] == '0') &
                   (df['RH_EX7'] == 'one'), 'VAR', df['D'])
df['D'].value_counts()

In [ ]:
108-97

In [ ]:
# 16, 17, 18, 19, 20 & 21 are all UND and individually unique, so leave for now

# 22 	NOAMP 	NOAMP 	0/0 	NOAMP 	zero 	N 	5
df['D'] = np.where((df['RHD_INT4'] =='NOAMP') &
                   (df['EX5'] == 'NOAMP') &
                   (df['RHD455'] == '0/0') & 
                   (df['RHD667'] == 'NOAMP') &
                   (df['RH_EX7'] == 'zero'), 'N', df['D'])
df['D'].value_counts()

In [ ]:
97-92
# DO THESE MENTALLY FROM NOW ON!

In [ ]:
# 23 und
# 24 	AMP 	1/1 	0/0 	NOAMP 	two 	N 	3
df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '1/1') &
                   (df['RHD455'] == '0/0') & 
                   (df['RHD667'] == 'NOAMP') &
                   (df['RH_EX7'] == 'two'), 'N', df['D'])
df['D'].value_counts()

In [ ]:
# 25 UND
# 26 	AMP 	0/1 	0/1 	0/0 	two 	VAR 	3
df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '0/1') &
                   (df['RHD455'] == '0/1') & 
                   (df['RHD667'] == '0/0') &
                   (df['RH_EX7'] == 'two'), 'VAR', df['D'])
df['D'].value_counts()

In [ ]:
# 27 	AMP 	0/1 	0/0 	1/1 	two 	VAR 	3
# 25	AMP	0/1	0/0	01-Jan	two	UND	3

df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '0/1') &
                   (df['RHD455'] == '0/0') & 
                   (df['RHD667'] == '1/1') &
                   (df['RH_EX7'] == 'two'), 'VAR', df['D'])
df['D'].value_counts()
# NOTE THIS CHANGED 6 (NOT 3) ROWS SO PICKED UP SOME UND ROWS - GOOD - THESE ARE FROM INDEX 25

In [ ]:
# 28 UND,
# 29 collected by index 9 combinations
# 30 & 31 are UND
# 32 collected by index 19 and SHOULD remain UND!!
# 32 AMP 	0 	0 	0/1 	one 	P 	2
# df['D'] = np.where((df['RHD_INT4'] =='AMP') &
#                   (df['EX5'] == '0') &
#                   (df['RHD455'] == '0') & 
#                   (df['RHD667'] == '0/1') &
#                   (df['RH_EX7'] == 'one'), 'P', df['D'])
# df['D'].value_counts()

In [ ]:
# 33, 34 & 35 are UND
# 36 	AMP 	0/0 	1/1 	1/1 	two 	VAR 	1
# 31	AMP	   0/0	   1/1	   1/1	   two	   UND	   2 
# NOTE 31 was skipped because UND, 
# happy this combionation is VAR be VAR
df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '0/0') &
                   (df['RHD455'] == '1/1') & 
                   (df['RHD667'] == '1/1') &
                   (df['RH_EX7'] == 'two'), 'VAR', df['D'])
df['D'].value_counts()

In [ ]:
# 37 collected by Index 1, 
# 38, 39 & 40 are UND
# 41 was collected by 12

In [ ]:
# 42 has the same genotype combination as Index 23
# 42 and 23 should remain UND as RHDpsi is 0/1 and Dcn is one. 
# 42 AMP 	0/1 	0 	0 	one 	P 	1
# df['D'] = np.where((df['RHD_INT4'] =='AMP') &
#                   (df['EX5'] == '0/1') &
#                   (df['RHD455'] == '0') & 
#                   (df['RHD667'] == '0') &
#                   (df['RH_EX7'] == 'one'), 'P', df['D'])
#df['D'].value_counts()

In [ ]:
# 43 and 44 UND
# 45 was collected by 33 and should remain UND
# AMP 	0/0 	0/0 	0/0 	zero 	P 	1
# df['D'] = np.where((df['RHD_INT4'] =='AMP') &
#                   (df['EX5'] == '0/0') &
#                   (df['RHD455'] == '0/0') & 
#                   (df['RHD667'] == '0/0') &
#                   (df['RH_EX7'] == 'zero'), 'P', df['D'])
# df['D'].value_counts()

In [ ]:
# 46 UND
# 47 collected by 10 and changed to VAR
# 48 	NOAMP 	NOAMP 	0/1 	NOAMP 	zero 	N 	1
df['D'] = np.where((df['RHD_INT4'] =='NOAMP') &
                   (df['EX5'] == 'NOAMP') &
                   (df['RHD455'] == '0/1') & 
                   (df['RHD667'] == 'NOAMP') &
                   (df['RH_EX7'] == 'zero'), 'N', df['D'])
df['D'].value_counts()

In [ ]:
# 49 was collected by 3
# 50, 51 are UND

In [ ]:
# complete for RHD

In [ ]:
# now check the 76 rows nan in ['D']
# NOTE this is down from 95 UND in RH01G - good.
# df['D'].value_counts()
#df[df['D'].isnan()]
#df[df['Col2'].isnull()]
df.dtypes

In [ ]:
df.head()

In [ ]:
df_Dnan = df[df['D'].str.contains ('nan')]
df_Dnan

In [ ]:
counts_Dnan = df_Dnan[['RHD_INT4', 'EX5', 'RHD455', 'RHD667', 'RH_EX7']].value_counts().reset_index(name='count')
counts_Dnan
# can I rationalise a phenotype for these genotypes?
# row 3 below shouldn't exist! RH_EX7 = one but RHD667 is 0/1 --> this combination should remain UND
# rows with NOAMP in EX5, RHD455 and/or RHD667 should remain UND 
#    as 1 or 1/1 calls would make Dvar
#    0 or 0/0 calls would make D+

In [ ]:
# row 2 in counts_Dnan 
# no interpretation made by the TT (DAT3577)
# manual interpretation suggests can be changed from UND to VAR
# 2 	AMP 	0 	0/1 	1 	one 	10

In [ ]:
# need to get the sample numbers for the 10 samples in index 2 above 
# and check Hematos to see if reported as D+VAR
index2_df = df[((df['RHD_INT4'] =='AMP') &
              (df['EX5'] == '0') &
                   (df['RHD455'] == '0/1') & 
                   (df['RHD667'] == '1') &
                   (df['RH_EX7'] == 'one'))]
index2_df

In [ ]:
# previously row 4 now 6
# need to chack Hematos to see if reported as VAR and then change form UND to VAR
# 6 	AMP 	0/0 	0/1 	1/1 	two 	5
index6_df = df[((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '0/0') &
                   (df['RHD455'] == '0/1') & 
                   (df['RHD667'] == '1/1') &
                   (df['RH_EX7'] == 'two'))]
index6_df

In [ ]:
# previoulsy row 11 now 16 in counts_Dnan 
# need to check Hematos to see if can be changed from UND to N
# 16 	AMP 	1 	0/1 	0 	one 	1
index16_df = df[((df['RHD_INT4'] =='AMP') &
                  (df['EX5'] == '1') &
                   (df['RHD455'] == '0/1') & 
                   (df['RHD667'] == '0') &
                   (df['RH_EX7'] == 'one'))]
index16_df

In [ ]:
# previously row 15 now index 18 in counts_Dnan
# need to check Hematos to see if can be changed from UND to VAR
# 18 	AMP 	0/0 	1/1 	0/1 	two 	1
index18_df = df[((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '0/0') &
                   (df['RHD455'] == '1/1') & 
                   (df['RHD667'] == '0/1') &
                   (df['RH_EX7'] == 'two'))]
index18_df

In [ ]:
# the above 4 combos (Index 2, 6, 16 and 18) have been cheked in Hematos 
# and the samples have been reported as per the manual interpretation.
# and can therefore be changed

In [ ]:
# row 2 in counts_Dnan is row 17 in counts_incRH01G
# no interpretation made by the TT (DAT3577)
# can be changed from UND to VAR
# 2 	AMP 	0 	0/1 	1 	one 	10

df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '0') &
                   (df['RHD455'] == '0/1') & 
                   (df['RHD667'] == '1') &
                   (df['RH_EX7'] == 'one'), 'VAR', df['D'])
df['D'].value_counts()

In [ ]:
# Index 6 in counts_Dnan 
# can be changed from UND to VAR
# 6 	AMP 	0/0 	0/1 	1/1 	two 	5
df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '0/0') &
                   (df['RHD455'] == '0/1') & 
                   (df['RHD667'] == '1/1') &
                   (df['RH_EX7'] == 'two'), 'VAR', df['D'])
df['D'].value_counts()

In [ ]:
# row 16 in counts_Dnan
# can be changed from UND to N
# 16 	AMP 	1 	0/1 	0 	one 	1
df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '1') &
                   (df['RHD455'] == '0/1') & 
                   (df['RHD667'] == '0') &
                   (df['RH_EX7'] == 'one'), 'N', df['D'])
df['D'].value_counts()

In [ ]:
# row 18 in counts_Dnan
# can be changed from UND to VAR
# 18 	AMP 	0/0 	1/1 	0/1 	two 	1
df['D'] = np.where((df['RHD_INT4'] =='AMP') &
                   (df['EX5'] == '0/0') &
                   (df['RHD455'] == '1/1') & 
                   (df['RHD667'] == '0/1') &
                   (df['RH_EX7'] == 'two'), 'VAR', df['D'])
df['D'].value_counts()

In [ ]:
print(df.shape)

In [ ]:
df_Dnonan = df[~df['D'].str.contains ('nan')]
print(df_Dnonan.shape)

In [ ]:
5854-5795

In [ ]:
df_Dnonan['D'].value_counts()

In [ ]:
counts_Dnonan = df_Dnonan[['RHD_INT4', 'EX5', 'RHD455', 'RHD667', 'RH_EX7', 'D']].value_counts().reset_index(name='count')
counts_Dnonan

In [ ]:
counts_Dnonan_plus1006 = df_Dnonan[['RHD_INT4', 'EX5', 'RHD455', 'RHD667', 'RH_EX7', 'D', 'RHCE1006']
].value_counts().reset_index(name='count')
counts_Dnonan_plus1006

In [ ]:
# counts_Dnonan.to_csv('HGP_D_data_v2', index=False)

In [ ]:
# counts_Dnonan_plus1006.to_csv('HGP_D_data_plus1006_v2', index=False)

In [ ]:
# df.to_csv('HGP_RH')

In [ ]:
df['D'].value_counts()

In [ ]:
df['RH01G'].value_counts()

In [ ]:
df[['D', 'RH01G']].value_counts()

In [ ]:
df['RH02G'].value_counts()

In [ ]:
df['RH05G'].value_counts()

In [ ]:
# df[['D', 'RH02G', 'RH05G']].value_counts()

counts_DCe = df[['D', 'RH02G', 'RH05G']
].value_counts().reset_index(name='count')
counts_DCe

In [ ]:
# count the number of rows with at least one 'VAR'in D C or e

((df['D'] == 'VAR') |
 (df['RH02G'] == 'VAR') |
 (df['RH05G'] == 'VAR')).sum()

In [ ]:
# df['RH01G'].str.contains('VAR') & df_project['RH02G'].str.contains('VAR').value_counts()

In [ ]:
# df[['RHC', 'RHE', 'RHCE667', 'RHCE1006', 'RHCE712', 'RHCE733', 'RH02G', 'RH03G', 'RH04G', 'RH05G']
# ].value_counts().reset_index(name='count')

In [ ]:
# check predicted pheno of C is correct
# at first glance index 5 and 8 below look wrong because the presence of RHCE1006
# usually associated C+VAR
df[['RHC', 'RHCE1006', 'RH02G', 'RH04G']
].value_counts().reset_index(name='count')

In [ ]:
# the below shows the C predicted phenotype is correct
# r'S cannot be present if the Dcn (RH_EX7) is two
CnotVAR = df[((df['RHC'] == '0/0') &
              (df['RHCE1006'] !='0/0') &
              (df['RH02G'] != 'VAR'))]
CnotVAR                

In [ ]:
print(CnotVAR.shape)

In [ ]:
# check predicted pheno of e is correct
df[['RHE', 'RHCE667', 'RHCE712', 'RHCE733', 'RH05G']
].value_counts().reset_index(name='count')

In [ ]:
# index 11 above suggests 667 change carried on E
# index 11 above same genotype but different phenotype to index 13
# get the sample numbers of affected samples

E667 = df[((df['RHE'] == '1/1') &
              (df['RHCE667'] !='0/0'))]
E667
# all the below should be set to e+VAR
# unlike D this is the only update required

In [ ]:
# Copy results of 'RH05G' to 'e'
df['e'] = df['RH05G']

In [ ]:
print(df.shape)

In [ ]:
# df['D'] = np.where((df['RHD_INT4'] =='AMP') &
#                   (df['EX5'] == '1') &
#                   (df['RHD455'] == '0/1') & 
#                   (df['RHD667'] == '0') &
#                   (df['RH_EX7'] == 'one'), 'N', df['D'])
# df['D'].value_counts()

df['e'] = np.where((df['RHE'] == '1/1') &
              (df['RHCE667'] !='0/0'), 'VAR', df['e'])

In [ ]:
# check predicted pheno of e is correct
df[['RHE', 'RH03G']
].value_counts().reset_index(name='count')

In [ ]:
4705+1054+95

In [ ]:
# check predicted pheno of e is correct
df[['RHC', 'RH04G']
].value_counts().reset_index(name='count')

In [ ]:
4127+1259+466+2

In [ ]:
HGP_RHCE_data = df[['RHC', 'RHE', 'RHCE667', 'RHCE1006', 'RHCE712', 'RHCE733', 'RH02G', 'RH03G', 'RH04G', 'e']
].value_counts().reset_index(name='count')

In [ ]:
# Do I need to check for Nan in RHCE????
# df_Dnonan = df[~df['D'].str.contains ('nan')]
# print(df_Dnonan.shape)

In [ ]:
HGP_RHCE_data

In [ ]:
# Now do it for all RH
HGP_RH_data = df[['RHD_INT4', 'EX5', 'RHD455', 'RHD667', 'RH_EX7', 'D', 
           'RHC', 'RHE', 'RHCE667', 'RHCE1006', 'RHCE712', 'RHCE733', 
           'RH02G', 'RH03G', 'RH04G', 'e']
].value_counts().reset_index(name='count')

In [ ]:
HGP_RH_data

In [ ]:
# count the number of rows with at least one 'VAR'in D RH02G or e

# HGP_RH_data = df[['RHD_INT4', 'EX5', 'RHD455', 'RHD667', 'RH_EX7', 'D', 
#           'RHC', 'RHE', 'RHCE667', 'RHCE1006', 'RHCE712', 'RHCE733', 
#           'RH02G', 'RH03G', 'RH04G', 'e']
# ].value_counts().reset_index(name='count')


# HGP_DCeVAR = df[['D'] == 'VAR', |
#  ['RH02G'] == 'VAR' |
#  ['RH05G'] == 'VAR'].value_counts().reset_index(name='count')

# Try making 3 new df
# then merge

df_DVAR = df[df['D'] == 'VAR']

In [ ]:
df_CVAR = df[df['RH02G'] == 'VAR']

In [ ]:
df_eVAR = df[df['e'] == 'VAR']

In [ ]:
print(df_DVAR.shape)

In [ ]:
# df_DVAR.to_csv('HGP_DVAR.csv',index=True)

In [ ]:
print(df_CVAR.shape)

In [ ]:
# df_CVAR.to_csv('HGP_CVAR.csv',index=True)

In [ ]:
print(df_eVAR.shape)

In [ ]:
# df_eVAR.to_csv('HGP_eVAR.csv',index=True)

In [ ]:
df_DCeVAR = pd.concat([df_DVAR, df_CVAR, df_eVAR], ignore_index=True, sort=False)
print(df_DCeVAR.shape)

In [ ]:
df_DCeVAR.drop_duplicates(subset=['Sample Number'], keep='first', inplace=True)

In [ ]:
print(df_DCeVAR.shape)

In [ ]:
# df_DCeVAR.to_csv('HGP_DCeVAR.csv',index=True)

In [ ]:
# HGP_DCeVAR.to_csv('HGP_DCeVAR.csv',index=True)

In [ ]:
# HGP_RHCE_data.to_csv('HGP_RHCE_data.csv',index=True)

In [ ]:
# HGP_RH_data.to_csv('HGP_RH_data.csv',index=True)

In [ ]:
# df.to_csv('HGP_all_clean_data_RH.csv',index=True)